# 04 - Granger Causality Tests

This notebook covers **Granger causality** testing for VAR models using `chronobox`.

## Topics covered

1. What is Granger causality?
2. Bivariate Granger causality tests (F-test and Wald test)
3. Multivariate Granger causality (all pairwise tests)
4. Directionality and feedback
5. Exercises

---

### Granger causality

Variable $X$ **Granger-causes** variable $Y$ if past values of $X$ contain information
that helps predict $Y$, beyond what is already contained in past values of $Y$ alone.

Formally, in a VAR(p) model, $X$ Granger-causes $Y$ if the coefficients on lagged $X$
in the equation for $Y$ are **jointly significantly different from zero**.

### The F-test

For the equation $Y_t = \sum_{i=1}^{p} a_i Y_{t-i} + \sum_{i=1}^{p} b_i X_{t-i} + u_t$:

$$H_0: b_1 = b_2 = \cdots = b_p = 0 \quad \text{(X does not Granger-cause Y)}$$

$$F = \frac{(RSS_r - RSS_u)/p}{RSS_u/(T - 2p - 1)} \sim F(p, T-2p-1)$$

### The Wald test

An asymptotically equivalent test:

$$W = T \cdot \frac{RSS_r - RSS_u}{RSS_u} \xrightarrow{d} \chi^2(p)$$

**Important:** Granger causality is about **predictive content**, not true economic causation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
from itertools import product as iterproduct

from chronobox import VAR
from chronobox.analysis.granger import granger_causality

sys.path.insert(0, os.path.join("..", "utils"))
from data_generators import generate_granger_causal

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Granger causality with synthetic data

Let's start with a controlled example where we **know** the true causal structure.
We generate data where variable $X$ Granger-causes $Y$ but not vice versa.

In [ ]:
# Generate data with known Granger-causal structure
# X -> Y with lag 2, coefficient 0.5
np.random.seed(42)
data_gc = generate_granger_causal(n=300, lag_effect=2, seed=42, coeff=0.5)

print(f"Generated data shape: {data_gc.shape}")

# Plot the two series
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(data_gc[:, 0], color="steelblue", linewidth=0.8)
axes[0].set_ylabel("X (cause)")
axes[0].set_title("Synthetic Granger-Causal Data")
axes[0].grid(True, alpha=0.3)

axes[1].plot(data_gc[:, 1], color="coral", linewidth=0.8)
axes[1].set_ylabel("Y (effect)")
axes[1].set_xlabel("Time")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Fit VAR and test Granger causality
gc_model = VAR(lags=4, trend="c")
gc_results = gc_model.fit(data_gc, names=["X", "Y"])

# Test: Does X Granger-cause Y?
gc_x_to_y = gc_results.granger_causality(caused="Y", causing="X", signif=0.05)
print("Test: X -> Y (expected: REJECT H0)")
print(f"  F-statistic: {gc_x_to_y.fstat:.4f}")
print(f"  F p-value:   {gc_x_to_y.pvalue:.6f}")
print(f"  Wald stat:   {gc_x_to_y.wald_stat:.4f}")
print(f"  Wald p-val:  {gc_x_to_y.wald_pvalue:.6f}")
print(f"  Degrees of freedom: {gc_x_to_y.df}")
print(f"  Reject H0 at 5%: {gc_x_to_y.reject}")

print()

# Test: Does Y Granger-cause X?
gc_y_to_x = gc_results.granger_causality(caused="X", causing="Y", signif=0.05)
print("Test: Y -> X (expected: FAIL TO REJECT H0)")
print(f"  F-statistic: {gc_y_to_x.fstat:.4f}")
print(f"  F p-value:   {gc_y_to_x.pvalue:.6f}")
print(f"  Wald stat:   {gc_y_to_x.wald_stat:.4f}")
print(f"  Wald p-val:  {gc_y_to_x.wald_pvalue:.6f}")
print(f"  Reject H0 at 5%: {gc_y_to_x.reject}")

**Result:** As expected, the test correctly identifies that X Granger-causes Y
(we reject $H_0$), but Y does not Granger-cause X (we fail to reject $H_0$).
This confirms the unidirectional causal structure we built into the data.

## 2. Granger causality with macroeconomic data

Now let's apply Granger causality tests to the Canadian macroeconomic dataset.
We test all pairwise relationships.

In [ ]:
# Load Canadian macro data and fit VAR
data_path = os.path.join("..", "data", "canada_macro.csv")
df = pd.read_csv(data_path)
var_names = ["e", "prod", "rw", "U"]
endog = df[var_names].values

model = VAR(lags=2, trend="c")
results = model.fit(endog, names=var_names)

print(f"VAR({results.k_ar}) fitted.\n")

In [ ]:
# All pairwise Granger causality tests
print("Pairwise Granger Causality Tests (5% significance)")
print("=" * 70)
print(f"{'Causing':>10s} -> {'Caused':>10s}  {'F-stat':>8s}  {'p-value':>10s}  {'Wald':>8s}  {'Decision':>10s}")
print("-" * 70)

gc_results_all = []
for causing in var_names:
    for caused in var_names:
        if causing == caused:
            continue
        gc = results.granger_causality(caused=caused, causing=causing, signif=0.05)
        decision = "REJECT" if gc.reject else "fail"
        print(f"{causing:>10s} -> {caused:>10s}  {gc.fstat:>8.3f}  {gc.pvalue:>10.4f}  {gc.wald_stat:>8.3f}  {decision:>10s}")
        gc_results_all.append({
            "causing": causing, "caused": caused,
            "fstat": gc.fstat, "pvalue": gc.pvalue,
            "wald_stat": gc.wald_stat, "reject": gc.reject
        })

In [ ]:
# Visualize Granger causality as a heatmap of p-values
gc_df = pd.DataFrame(gc_results_all)
pval_matrix = gc_df.pivot(index="caused", columns="causing", values="pvalue")

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pval_matrix.values, cmap="RdYlGn", vmin=0, vmax=0.2, aspect="auto")

ax.set_xticks(range(len(pval_matrix.columns)))
ax.set_xticklabels(pval_matrix.columns)
ax.set_yticks(range(len(pval_matrix.index)))
ax.set_yticklabels(pval_matrix.index)
ax.set_xlabel("Causing variable")
ax.set_ylabel("Caused variable")
ax.set_title("Granger Causality P-Values\n(green = significant, red = not significant)")

# Annotate cells
for i in range(len(pval_matrix.index)):
    for j in range(len(pval_matrix.columns)):
        val = pval_matrix.values[i, j]
        if not np.isnan(val):
            star = "***" if val < 0.01 else "**" if val < 0.05 else "*" if val < 0.10 else ""
            ax.text(j, i, f"{val:.3f}{star}", ha="center", va="center", fontsize=10)

plt.colorbar(im, ax=ax, label="p-value")
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "granger_heatmap.png"), bbox_inches="tight")
plt.show()

**Economic interpretation:**

The Granger causality heatmap reveals the predictive relationships in the Canadian
macroeconomy:
- Significant results (green cells) indicate that the causing variable has predictive
  content for the caused variable beyond its own history.
- **Bidirectional feedback** occurs when both $X \to Y$ and $Y \to X$ are significant.
- **Unidirectional causality** occurs when only one direction is significant.

Remember: Granger causality tests **predictive content**, not structural causation.

## 3. Directionality and feedback analysis

In [ ]:
# Classify relationships
print("Causal Structure Summary")
print("=" * 50)

pairs_tested = set()
for i, v1 in enumerate(var_names):
    for j, v2 in enumerate(var_names):
        if i >= j:
            continue
        pair = (v1, v2)
        if pair in pairs_tested:
            continue
        pairs_tested.add(pair)

        gc_1to2 = results.granger_causality(caused=v2, causing=v1, signif=0.05)
        gc_2to1 = results.granger_causality(caused=v1, causing=v2, signif=0.05)

        if gc_1to2.reject and gc_2to1.reject:
            print(f"  {v1} <-> {v2}  (FEEDBACK: bidirectional)")
        elif gc_1to2.reject:
            print(f"  {v1}  -> {v2}  (unidirectional)")
        elif gc_2to1.reject:
            print(f"  {v1} <-  {v2}  (unidirectional)")
        else:
            print(f"  {v1}  x  {v2}  (no Granger causality)")

In [ ]:
# Network visualization of causal structure
fig, ax = plt.subplots(figsize=(8, 8))

# Position variables in a circle
n_vars = len(var_names)
angles = np.linspace(0, 2 * np.pi, n_vars, endpoint=False)
radius = 2.0
positions = {name: (radius * np.cos(a), radius * np.sin(a)) for name, a in zip(var_names, angles)}

# Draw nodes
for name, (x, y) in positions.items():
    ax.scatter(x, y, s=2000, c="lightsteelblue", edgecolors="navy", linewidths=2, zorder=5)
    ax.text(x, y, name, ha="center", va="center", fontsize=14, fontweight="bold", zorder=6)

# Draw edges (arrows) for significant Granger causality
for causing in var_names:
    for caused in var_names:
        if causing == caused:
            continue
        gc = results.granger_causality(caused=caused, causing=causing, signif=0.05)
        if gc.reject:
            x1, y1 = positions[causing]
            x2, y2 = positions[caused]
            # Shorten arrow to not overlap with node
            dx, dy = x2 - x1, y2 - y1
            length = np.sqrt(dx**2 + dy**2)
            shrink = 0.55 / length
            ax.annotate("", xy=(x2 - dx * shrink, y2 - dy * shrink),
                       xytext=(x1 + dx * shrink, y1 + dy * shrink),
                       arrowprops=dict(arrowstyle="->", color="red", lw=2,
                                      connectionstyle="arc3,rad=0.15"))

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect("equal")
ax.set_title("Granger Causality Network (5% significance)", fontsize=14)
ax.axis("off")
plt.savefig(os.path.join("..", "outputs", "granger_network.png"), bbox_inches="tight")
plt.show()

## 4. Sensitivity to significance level and lag order

In [ ]:
# Test sensitivity to significance level
print("Sensitivity to significance level:")
print(f"{'Pair':>15s}  {'p-value':>10s}  {'1%':>6s}  {'5%':>6s}  {'10%':>6s}")
print("-" * 55)

for causing in var_names:
    for caused in var_names:
        if causing == caused:
            continue
        gc = results.granger_causality(caused=caused, causing=causing)
        r01 = "Yes" if gc.pvalue < 0.01 else "No"
        r05 = "Yes" if gc.pvalue < 0.05 else "No"
        r10 = "Yes" if gc.pvalue < 0.10 else "No"
        print(f"{causing+' -> '+caused:>15s}  {gc.pvalue:>10.4f}  {r01:>6s}  {r05:>6s}  {r10:>6s}")

In [ ]:
# Sensitivity to lag order: test e -> U across different VAR orders
print("Effect of lag order on Granger causality (e -> U):")
print(f"{'Lag':>4s}  {'F-stat':>8s}  {'p-value':>10s}  {'Reject 5%':>10s}")
print("-" * 40)

pvals_by_lag = []
for p in range(1, 9):
    m = VAR(lags=p, trend="c")
    r = m.fit(endog, names=var_names)
    gc = r.granger_causality(caused="U", causing="e", signif=0.05)
    decision = "REJECT" if gc.reject else "fail"
    print(f"{p:>4d}  {gc.fstat:>8.3f}  {gc.pvalue:>10.4f}  {decision:>10s}")
    pvals_by_lag.append(gc.pvalue)

# Plot p-values by lag
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 9), pvals_by_lag, "o-", color="steelblue", linewidth=1.5)
ax.axhline(0.05, color="red", linestyle="--", label="5% significance")
ax.axhline(0.01, color="orange", linestyle="--", label="1% significance")
ax.set_xlabel("VAR lag order")
ax.set_ylabel("p-value")
ax.set_title("Granger Causality p-value: e -> U (by lag order)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "granger_lag_sensitivity.png"), bbox_inches="tight")
plt.show()

**Economic interpretation:**

- Granger causality results can be sensitive to the lag order and significance level.
- This is why proper lag selection (using information criteria) is important *before*
  running Granger causality tests.
- Results should be interpreted alongside economic theory, not in isolation.
- Bidirectional feedback between variables is common in macroeconomics -- for example,
  employment and unemployment naturally influence each other.

## Exercicio

### Exercicio 1: Causalidade de Granger nos dados US Macro

Use o dataset `us_macro_quarterly.csv` para:

1. Ajustar um VAR(4)
2. Testar todas as relacoes de Granger pairwise
3. Criar o heatmap de p-values
4. Responder: O Fed Funds Rate Granger-causa GDP? E inflacao? Existem relacoes de feedback?

**Dica:** Relacoes significativas entre `fed_funds` e outras variaveis sao evidencia
de que a politica monetaria tem impacto preditivo na economia.

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

us_df = pd.read_csv(os.path.join("..", "data", "us_macro_quarterly.csv"))
us_names = ["gdp", "inflation", "fed_funds", "unemployment"]
us_endog = us_df[us_names].values

us_model = VAR(lags=4, trend="c")
us_results = us_model.fit(us_endog, names=us_names)

print("US Macro Granger Causality (5% significance)")
print("=" * 70)

us_gc_all = []
for causing in us_names:
    for caused in us_names:
        if causing == caused:
            continue
        gc = us_results.granger_causality(caused=caused, causing=causing, signif=0.05)
        us_gc_all.append({"causing": causing, "caused": caused,
                          "pvalue": gc.pvalue, "reject": gc.reject})
        star = "***" if gc.pvalue < 0.01 else "**" if gc.pvalue < 0.05 else ""
        print(f"  {causing:>12s} -> {caused:<12s}  p={gc.pvalue:.4f} {star}")

# Heatmap
us_gc_df = pd.DataFrame(us_gc_all)
us_pval = us_gc_df.pivot(index="caused", columns="causing", values="pvalue")

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(us_pval.values, cmap="RdYlGn", vmin=0, vmax=0.2, aspect="auto")
ax.set_xticks(range(len(us_pval.columns)))
ax.set_xticklabels(us_pval.columns, rotation=45)
ax.set_yticks(range(len(us_pval.index)))
ax.set_yticklabels(us_pval.index)
ax.set_xlabel("Causing")
ax.set_ylabel("Caused")
ax.set_title("Granger Causality P-Values - US Macro")

for i in range(len(us_pval.index)):
    for j in range(len(us_pval.columns)):
        val = us_pval.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=10)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Exercicio 2: Granger causality com dados sinteticos

Gere dados sinteticos com a funcao `generate_granger_causal` usando diferentes
parametros:

1. Varie o coeficiente (`coeff=0.1, 0.3, 0.5, 0.8`)
2. Para cada coeficiente, rode o teste de Granger
3. Plote p-value vs coeficiente
4. A partir de qual coeficiente o teste detecta causalidade de forma consistente?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

coefficients = [0.05, 0.1, 0.2, 0.3, 0.5, 0.8]
pvals = []

for c in coefficients:
    data_syn = generate_granger_causal(n=200, lag_effect=2, seed=42, coeff=c)
    m = VAR(lags=4, trend="c")
    r = m.fit(data_syn, names=["X", "Y"])
    gc = r.granger_causality(caused="Y", causing="X")
    pvals.append(gc.pvalue)
    print(f"coeff={c:.2f}: F={gc.fstat:.3f}, p={gc.pvalue:.4f}, reject={gc.reject}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(coefficients, pvals, "o-", color="steelblue", linewidth=1.5)
ax.axhline(0.05, color="red", linestyle="--", label="5% significance")
ax.set_xlabel("Causal coefficient")
ax.set_ylabel("p-value")
ax.set_title("Detection Power vs. Effect Size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Resumo

Neste notebook aprendemos:

1. **Causalidade de Granger** testa se os valores passados de uma variavel melhoram a previsao de outra
2. O **teste F** e o **teste de Wald** fornecem estatisticas para a hipotese nula de nao-causalidade
3. Relacoes **bidirecionais** (feedback) sao comuns em dados macroeconomicos
4. Os resultados sao **sensiveis** a ordem de lags e nivel de significancia
5. Causalidade de Granger mede **conteudo preditivo**, nao causalidade estrutural
6. O **poder do teste** aumenta com o tamanho do efeito e o tamanho da amostra

### Serie completa de notebooks VAR/VECM:
- **01**: Introducao ao VAR (selecao de ordem, estabilidade, previsao)
- **02**: VECM e teste de Johansen (cointegracao, alpha, beta)
- **03**: IRF e FEVD (funcoes impulso-resposta, decomposicao de variancia)
- **04**: Causalidade de Granger (testes bivariados e multivariados)